In [ ]:
# === PARAMETERS ===
MODEL_TYPE = "MLP" # "MLP" or "CNN" or "COORD"
LOG_FILE = "training_log.txt"
NAME_ID = ""

# Grid / Agents
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# MLP Configuration
MLP_HIDDEN_DIM = 512
MLP_NUM_LAYERS = 4

# CNN Configuration
CNN_CONV_CHANNELS = [64, 64, 64]
CNN_HEAD_HIDDEN_DIM = 64
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

# Training
LR = 0.0001
BATCH_SIZE = 64
DISCOUNT = 0.0 # Immediate Reward (Regression)
STOP_THRESHOLD_MAE = 0.01
SEED = 42
MAX_STEPS = 20000000 # Reward learning converges fast
EVAL_FREQ = 1000

NUM_TEST_SAMPLES_PER_TYPE = 2000

In [8]:
OUT_FOLDER = f"reward_centralized_{MODEL_TYPE}_lr{LR}_{NAME_ID}"

In [9]:
from pathlib import Path
import sys
import numpy as np
import torch
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add root to path (assuming notebook is in experiment folder)
sys.path.append("../") 

from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy
from models.value_mlp import ValueMLPCentralized
from models.value_cnn_new import ValueCNNCentralizedStandard, ValueCNNCoord
import ast

if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)
    
# Initial Seed (Will be reset in loop)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

OUT_FOLDER = Path(OUT_FOLDER)
OUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT_FOLDER / LOG_FILE

In [10]:
# --- INITIALIZATION LOGIC ---
if MODEL_TYPE == "MLP":
    model = ValueMLPCentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
    )
elif MODEL_TYPE == "CNN":
    # The Deep Standard CNN (Professor Approved: Flatten, No Coords)
    model = ValueCNNCentralizedStandard(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
elif MODEL_TYPE == "COORD":
    # The Sanity Check (CoordConv: Tunable)
    # Note: You can force kernel_size=1 here if you want strict "Pure CoordConv"
    # or use CNN_KERNEL_SIZE to see if 3x3 + Coords helps.
    model = ValueCNNCoord(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, 
        num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, 
        kernel_size=CNN_KERNEL_SIZE
    )
else:
    raise ValueError(f"Invalid MODEL_TYPE: {MODEL_TYPE}")

params = sum(p.numel() for p in model.policy_net.parameters())
print(f"Initialized {MODEL_TYPE}. Total Trainable Parameters: {params}")

Initialized MLP. Total Trainable Parameters: 871937


In [11]:
def generate_balanced_test_suite(n):
    states_zero = []
    states_pick = []
    
    # 1. Zeros (No agent on ANY apple)
    while len(states_zero) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        # Add noise apples STRICTLY on empty tiles
        num_noise = np.random.randint(1, 6)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
        states_zero.append(s)
        
    # 2. Pickers (Exactly ONE agent on ONE apple)
    while len(states_pick) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        s.apples[:] = 0
        
        # A. Pick Target
        picker_id = np.random.randint(0, NUM_AGENTS)
        r_t, c_t = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
        
        # B. Place Picker and Apple
        s.set_agent_position(picker_id, np.array([r_t, c_t]))
        s.apples[r_t, c_t] = 1
        
        # C. Ensure NO other agent is at (r_t, c_t)
        for other_id in range(NUM_AGENTS):
            if other_id != picker_id:
                while tuple(s.agent_position(other_id)) == (r_t, c_t):
                    r_new, c_new = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
                    s.set_agent_position(other_id, np.array([r_new, c_new]))

        # D. Add Noise Apples (Strictly on empty tiles)
        num_noise = np.random.randint(0, 5)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
            
        states_pick.append(s)
        
    return states_zero, states_pick

test_zeros, test_picks = generate_balanced_test_suite(NUM_TEST_SAMPLES_PER_TYPE)
print(f"Test Suite: {len(test_zeros)} Zeros, {len(test_picks)} Pickers")

Test Suite: 2000 Zeros, 2000 Pickers


In [12]:
# --- TRAINING LOOP ---\n
loss_history = []
max_ape_history = []
soft_convergence_step = None # Track when we first hit Avg < 1%

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Reset Buffer
model.memory.memory.clear()

s: State = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)

# Write Header
with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING BENCHMARK: {MODEL_TYPE} [LR={LR}] ===\n")
    f.write("Step   | Loss    | Max%_Z | Avg%_Z | Max%_P | Avg%_P | Status\n")

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    
    # 1. MOVE (Reward 0)
    active_agent_idx = s.get_random_agent_id()
    move_action = nearest_policy(s, active_agent_idx)
    new_pos = np.clip(s.agent_position(active_agent_idx) + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    model.add_experience(s, s_mid, 0.0)

    # 2. PICK (Reward 1.0 or 0.0)
    s_next = s_mid.copy()
    reward = 0.0
    if s_next.apples[tuple(new_pos)] > 0:
        s_next.remove_apple_at(new_pos)
        reward = 1.0 # Target is 1.0
        
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    if reward != 0:
        model.add_experience(s_mid, s_next, reward)
    else:
        model.add_experience(s_mid, s_next, 0.0)

    # 3. TRAIN
    loss = model.train_batch(BATCH_SIZE)
    if loss is not None: loss_history.append(loss)
    s = s_next
    
    # 4. EVAL
    if step % EVAL_FREQ == 0 and step > 0:
        # Evaluate Zeros
        preds_z = np.array([model.get_value(st) for st in test_zeros])
        if len(preds_z) > 0:
            max_ape_z = np.max(np.abs(preds_z)) * 100
            avg_ape_z = np.mean(np.abs(preds_z)) * 100
        else: max_ape_z, avg_ape_z = 0, 0
        
        # Evaluate Pickers (Target 1.0)
        preds_p = np.array([model.get_value(st) for st in test_picks])
        if len(preds_p) > 0:
            max_ape_p = np.max(np.abs(preds_p - 1.0)) * 100
            avg_ape_p = np.mean(np.abs(preds_p - 1.0)) * 100
        else: max_ape_p, avg_ape_p = 0, 0
        
        # Check Metrics
        global_max = max(max_ape_z, max_ape_p)
        global_avg = max(avg_ape_z, avg_ape_p)
        
        status_msg = ""
        
        # A. Soft Convergence Check (Avg < 1%)
        if global_avg < 1.0:
            if soft_convergence_step is None:
                soft_convergence_step = step
                print(f"\n🌟 SOFT CONVERGENCE (Avg < 1%) reached at Step {step}!")
            status_msg = "AVG_OK"

        # B. Strict Convergence Check (Max < 1%)
        if global_max < 1.0:
            status_msg = "STRICT_OK"
        
        # Log
        pct_z = 0
        if len(model.memory) > 0:
            r = [t.reward for t in model.memory.memory]
            pct_z = (r.count(0.0) / len(r)) * 100

        with open(LOG_FILE, "a") as f:
            f.write(f"{step:<6} | {loss:.5f} | {max_ape_z:6.2f} | {avg_ape_z:6.2f} | {max_ape_p:6.2f} | {avg_ape_p:6.2f} | {status_msg}\n")
            
        pbar.set_description(f"L:{loss:.4f} | Max%:{global_max:.0f}% | Avg%:{global_avg:.1f}%")
        
        # C. STOPPING CONDITION
        if global_max < 1.0:
            print(f"✅ STRICT CONVERGENCE (Max < 1%) at Step {step}")
            break

  0%|          | 0/20000000 [00:00<?, ?it/s]

L:0.0824 | Max%:124% | Avg%:83.1%:   0%|          | 1954/20000000 [00:13<39:26:08, 140.86it/s]


KeyboardInterrupt: 